# ORCA-X v2.6 — Kaggle GPU Training + Live Data

Controlled GPU entry point for the forward-6-hour model. GPU = Tesla T4 is sufficient; Internet must be ON. Historical evaluation remains locked and production promotion is disabled.

Live supervised training is horizon-aware: a live observation becomes a training row only after its +6h future observation has matured.

In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/Sayan260106/HackHeritage.git'
REPO_REF = 'main'
REPO_DIR = Path('/kaggle/working/HackHeritage')
os.environ['ORCA_X_DEVICE'] = 'cuda'
os.environ['ORCA_X_N_JOBS'] = '2'
os.environ['ORCA_PROMOTE_MODEL'] = 'false'
print('Branch:', REPO_REF)
print('ORCA_X_DEVICE:', os.environ['ORCA_X_DEVICE'])
print('ORCA_X_N_JOBS:', os.environ['ORCA_X_N_JOBS'])
print('ORCA_PROMOTE_MODEL:', os.environ['ORCA_PROMOTE_MODEL'])

In [ ]:
%cd /kaggle/working
!rm -rf HackHeritage
!git clone --depth 1 --branch main --single-branch "{REPO_URL}" HackHeritage
%cd /kaggle/working/HackHeritage
!git rev-parse --abbrev-ref HEAD
!git rev-parse HEAD

In [ ]:
!python -m pip install -q --upgrade pip
!python -m pip install -q -r ml/requirements-colab.txt
import torch, xgboost as xgb
print('PyTorch CUDA available:', torch.cuda.is_available())
print('XGBoost version:', xgb.__version__)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
DATA = Path('/kaggle/working/HackHeritage/ml/data/processed/orca_historical_marine_risk.parquet')
if not DATA.exists():
    !python ml/src/colab_prepare.py
assert DATA.exists(), DATA
print('Dataset ready:', DATA)

In [ ]:
print('Running training preflight...')
!python ml/src/training_preflight.py

In [ ]:
print('Running v2.6 evaluation-only training...')
!python ml/src/colab_gpu_runner.py ml/src/train.py

In [ ]:
print('Running Refinement 40 error analysis + production gate...')
!python ml/src/refinement40_error_analysis.py

## Live-data training

Run `collect` on an hourly schedule. After at least six hours, run `mature`. Once enough matured rows accumulate, run `train-candidate`. The candidate is saved separately and never overwrites the production artifact.

In [ ]:
!python ml/src/realtime_training.py collect

In [ ]:
!python ml/src/realtime_training.py mature

In [ ]:
!python ml/src/realtime_training.py train-candidate